In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import sys, os, re
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.ticker as ticker  
import cartopy.feature as cfeature
import cartopy.io.shapereader as shapereader
import matplotlib.ticker as mticker
from collections import defaultdict
from scipy.stats import gaussian_kde
sys.path.append(os.path.abspath(".."))
from models.utils import population as pop
import models.Burkart2022.model.PAF_calculations as paf
import comparison.utils as utils
from matplotlib.ticker import FuncFormatter
from collections import defaultdict

### Data analysis

In [ ]:
### AR6 pathways
ar6 = pd.read_csv("X:\\user\\dekkerm\\Data\\AR6_snapshots\\AR6_snapshot_Nayeli_global.csv")
cc_path_mean = ar6.iloc[:, 10:-1].groupby(ar6["Category"]).mean()
cc_path_std = ar6.iloc[:, 10:-1].groupby(ar6["Category"]).std()

original_colors = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7"]
fig, ax = plt.subplots(figsize=(8, 5))
for i,categories in enumerate(ar6["Category"].unique()):
    plt.plot(cc_path_mean.columns, cc_path_mean.loc[categories], label=categories, color=original_colors[i])
    plt.fill_between(cc_path_mean.columns, cc_path_mean.loc[categories] - 2*cc_path_std.loc[categories], cc_path_mean.loc[categories] + 2*cc_path_std.loc[categories], 
                     color=original_colors[i], alpha=0.2)
ax.set_xticks(cc_path_mean.columns[::10])
utils.StylizePlot(title="CC pathways - AR6 \n(2std uncertainty range)", ylabel="GMST anomaly (°C)", xlabel="Year", legend=True, show=True)

#### Temperature Zones

In [ ]:
wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
era5_tz = xr.open_dataset(f"{wdir}/data/temperature_zones/ERA5_mean_1980-2019_land_t2m_tz.nc")
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(111, projection=ccrs.Robinson(central_longitude=0), frameon=True)
ax.coastlines(resolution='10m', lw = 0.1)
ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)
ax.spines['geo'].set_linewidth(0.4)

# Gridlines
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False, linewidth=0, color='lightgray', alpha=0.5)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 8}
gl.ylabel_style = {'size': 8}

# Plot the temperature zones
im = era5_tz.t2m.plot(
    ax=ax, 
    transform=ccrs.PlateCarree(), 
    cbar_kwargs = {
        "orientation": "horizontal", 
        "shrink":0.5, 
        "pad":0.06, 
        "aspect":20, 
        "label":"Temperature zones", 
        "ticks": [6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28]
        },
    vmin=6,
    vmax=28, 
    levels=23,
    extend="both",
    cmap = "cividis")

# Remove Antarctica 
land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    if land.bounds[1] < -60:
        ax.add_geometries([land], ccrs.PlateCarree(), 
                          facecolor='whitesmoke', 
                          edgecolor='none', 
                          zorder=2)

plt.savefig(os.path.dirname(wdir) +"/figures/Paper1/Burkart_TemperatureZones.png", dpi=300, bbox_inches='tight')
plt.show()

#### TMRELs

##### Population weighted PDFs of TMRELs vs Temperature Zones

In [ ]:
### Open all files and calculate KDE

wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
years = [1990,2010,2020]

# Import population maps for years with TMRELs
population = pop.LoadPopulationMap(wdir, scenario="SSP2_ERA5", ssp="SSP2", years=[1990,2020])
population = population.sel(time=["1990", "2010", "2020"]).mean("time")

# Import TMRELs maps
tmrel = {}
for year in years:
    tmrel[year] = xr.open_dataset(wdir+f"/data/TMRELs_nc/TMRELs_{year}.nc")

# Import temperature zones
era5_tz = xr.open_dataset(wdir+f"/data/temperature_zones/ERA5_mean_1980-2019_land_t2m_tz.nc")

# Initialize dicitonaries for counts and weights for KDE
counts_tmrel = defaultdict(dict); weights_tmrel = defaultdict(dict)

for year in years:
    for tz in range(6,29):
        counts_tmrel[year][tz], weights_tmrel[year][tz] = utils.WeightsAndCountsForKDE(tmrel[year], era5_tz, population,tz)

In [ ]:
### Plot TMRELs vs TZ for one year
year = 2010
tz_values = list(range(6, 29))
colors = cm.copper(np.linspace(0, 1, len(tz_values)))

fig, ax = plt.subplots(figsize=(10, 8))#, dpi=300)

for i, tz in enumerate(tz_values):
    data = counts_tmrel[year][tz]
    w = weights_tmrel[year][tz]

    if len(data) > 50000:
        idx = np.random.choice(len(data), size=50000, replace=False, p=w)
        data = data[idx]
        w = w[idx]
        w = w / w.sum()

    kde = gaussian_kde(data, weights=w)

    p5, p95 = np.percentile(data, [5, 95])
    x = np.linspace(p5, p95, 200)
    y = kde(x)

    scale = 0.4 / y.max()
    y_scaled = y * scale

    ax.fill_betweenx(x, tz, tz + y_scaled, color=colors[i], alpha=0.8, zorder=2)

utils.StylizeAxes(ax, xticks=tz_values, xtickslabels=tz_values, xlim=(5.5,28.5), ylim=(14,30), xlabel="Temperature zone [°C]",
             ylabel="TMREL [°C]", title=f"Population-weighted TMREL distributions by Temperature Zone for {year}\n5–95 Percentile Range\n",
             grid=False)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

In [ ]:
### Plot TMRELs vs TZ for all years

tz_values = list(range(6, 29))
colors = cm.copper(np.linspace(0, 1, len(tz_values)))

fig, axs = plt.subplots(1,3, figsize=(18, 5))#, dpi=300)
axs=axs.flatten()

for j,year in enumerate([1990,2010,2020]):
    for i, tz in enumerate(tz_values):
        data = counts_tmrel[year][tz]
        w = weights_tmrel[year][tz]

        if len(data) > 50000:
            idx = np.random.choice(len(data), size=50000, replace=False, p=w)
            data = data[idx]
            w = w[idx]
            w = w / w.sum()

        kde = gaussian_kde(data, weights=w)

        p5, p95 = np.percentile(data, [5, 95])
        x = np.linspace(p5, p95, 200)
        y = kde(x)

        scale = 0.4 / y.max()
        y_scaled = y * scale

        axs[j].fill_betweenx(x, tz, tz + y_scaled, color=colors[i], alpha=0.9, zorder=2)

    utils.StylizeAxes(axs[j], xticks=tz_values, xtickslabels=tz_values, xlim=(5.5,28.5), ylim=(13.5,30), xlabel="Temperature zone [°C]",
                ylabel="TMREL [°C]", title=f"{year}", grid=True, grid_kwargs={"axis":"both", "color":"white", "zorder":0}, facecolor="whitesmoke")
plt.suptitle("Population-weighted TMREL Distributions by Temperature Zone")

#### ERFs

In [ ]:
### Plotting relevant causes mean of RR

sets = paf.ModelSettings(
        wdir="X:/user/liprandicn/mt-comparison/burkart2022",
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        project=None,
        scenario="SSP2_ERA5",
        years=[2010],
        draw="mean",
        single_erf=False, 
        extrap_erf=False,
    )

erf, _, _ = paf.LoadExposureResponseFunctions(sets)
tmrel = paf.LoadTMRELsMap(sets, 2010)


plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.magma(np.linspace(0,1,23)))

fig, axs = plt.subplots(2,4, figsize=(18,8))#, dpi=300)
axs = axs.flatten()

for i, disease in enumerate(utils.relevant_causes.keys()):
    for tz in range(6,29):
        
        x = erf["daily_temperature"].loc[tz].values   # Daily Temperature
        y = erf[disease].loc[tz].values    # Relative Risk
        
        # x_shift = x - x[np.argmin(y)] # Horizontal shift
        y_shift = y - np.min(y) + 1   # Move the minimum to 1 RR
        
        axs[i].plot(x, y_shift, label=tz)

    utils.StylizeAxes(axs[i], facecolor="whitesmoke", grid=True, grid_kwargs={"c":"white"}, title=f"{utils.relevant_causes[disease]}", 
                      spines={"top":False, "right":False, "left":False, "bottom":False})#, ylim=(0.9,2))
    
    for i in [0,4]:
        utils.StylizeAxes(axs[i], ylabel="Relative Risk")
    for i in range(4,8):
        utils.StylizeAxes(axs[i], xlabel="Daily mean temperature [°C]")
    
plt.legend(loc="upper center", bbox_to_anchor=(-1.3, -0.2), ncol=8, title = "Temperature zones")

In [ ]:
### Plotting All causes and referred  to TMRELs

wdir="X:/user/liprandicn/mt-comparison/burkart2022"

sets = paf.ModelSettings(
        wdir=wdir,
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        project=None,
        scenario="SSP2_ERA5",
        years=[2010],
        draw="mean",
        single_erf=False, 
        extrap_erf=False,
    )

erf = paf.LoadExposureResponseFunctionsAll(sets)


for cause in utils.causes.keys():
    erf[cause] = erf[cause].set_index("temperature_zone")
    erf[cause]["mean"] = erf[cause].iloc[:,1:].mean(axis=1)

plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.cividis(np.linspace(0,1,23)))

fig, axs = plt.subplots(5,4, figsize=(12,12))#, dpi=300)
axs = axs.flatten()

for i, cause in enumerate(utils.causes.keys()):
    
    for tz in range(6,29):
        
        x = erf[cause]["daily_temperature"].loc[tz].values   # Daily Temperature
        y = erf[cause]["mean"].loc[tz].values    # Relative Risk        
        axs[i].plot(x, y, label=tz)
        axs[i].axhline(y=1, color="grey", linestyle="--", linewidth=0.8, zorder=0)

    utils.StylizeAxes(
        axs[i],
        facecolor="whitesmoke",
        grid=True, 
        grid_kwargs={"c":"white"}, 
        title=f"{utils.causes[cause]}", 
        title_kwargs={"fontsize":10, "fontweight":"bold"}, 
        spines={"top":False, "right":False, "left":False, "bottom":False},
        ylim=(0.75,2)
        )
    
    for i in [0,4, 8, 12, 16]:
        utils.StylizeAxes(
            axs[i],
            ylabel="Relative Risk [RR]",
            ylabel_kwargs={"fontsize":8}
            )
    for i in range(13,17):
        utils.StylizeAxes(
            axs[i],
            xlabel="Daily mean temperature [°C]",
            xlabel_kwargs={"fontsize":8}
            )
        
for i in [17, 18, 19]:
    axs[i].axis('off')


plt.tight_layout()
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.65, 0.15), ncol=8, title="Temperature zones", title_fontsize=10)

plt.savefig(os.path.dirname(wdir) +"/figures/Paper1/Burkart_ERFs.png", dpi=300, bbox_inches='tight')

In [ ]:
### Plotting EXTRAPOLATION of ERFs
sets = paf.ModelSettings(
        wdir="X:/user/liprandicn/mt-comparison/burkart2022",
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        project=None,
        scenario="SSP2_ERA5",
        years=[2010],
        draw="mean",
        single_erf="False", 
        extrap_erf=True,
    )

erf_ex, _, _ = paf.LoadExposureResponseFunctions(sets)

plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.magma(np.linspace(0,1,23)))

fig, axs = plt.subplots(2,4, figsize=(18,8))#, dpi=300)
axs = axs.flatten()

for i, disease in enumerate(utils.relevant_causes.keys()):
    for tz in range(6,29):
        
        # --- Original data ---
        x_orig = erf["daily_temperature"].loc[tz].values
        y_orig = erf[disease].loc[tz].values
        y_orig_shift = y_orig - np.min(y_orig) + 1  # Vertical shift
        x_orig_shift = x_orig - x_orig[np.argmin(y_orig)] # Horizontal shift
        axs[i].plot(x_orig_shift, y_orig_shift, label=tz, linestyle="-")  # Solid line

        # --- Extrapolated data ---
        x_ex = erf_ex["daily_temperature"].loc[tz].values
        y_ex = erf_ex[disease].loc[tz].values
        y_ex_shift = y_ex - np.min(y_ex) + 1  # Vertical shift
        x_ex_shift = x_ex - x_ex[np.argmin(y_ex)] # Horizontal shift
        axs[i].plot(x_ex_shift, y_ex_shift, linestyle="--")  # Dashed line

    utils.StylizeAxes(axs[i], facecolor="whitesmoke", grid=True, grid_kwargs={"c":"white"}, title=f"{utils.relevant_causes[disease]}", ylim=(0.9,5))
    
    for i in [0,4]:
        utils.StylizeAxes(axs[i], ylabel="Relative Risk")
    for i in range(4,8):
        utils.StylizeAxes(axs[i], xlabel="Daily temperature [°C]")
    
plt.legend(loc="upper center", bbox_to_anchor=(-1.3, -0.2), ncol=8, title = "Temperature zones")

### Results

#### Replication results

In [ ]:
# Stacked bar plot

wdir = "X:/user/liprandicn/mt-comparison/burkart2022"
b19 = pd.read_csv(wdir+"/output/ReplicationBurkart/MOR_ReplicationBurkart_SSP2_ERA5_IMAGE_1990-2019.csv")

age_group = "All ages"
t_type = "all"
year = "2019"

b19 = b19[(b19["age_group"]==age_group) & 
    (b19["t_type"]==t_type) & 
    (b19["units"]=="Total Mortality") & 
    (~b19["region"].isin(["GRL", "World"])) &
    (b19["cause"] != "All causes")].drop(columns=["age_group", "units", "t_type"])[["region", "cause", year]]
b19=b19.pivot(index="region", columns="cause", values=year)
b19.index = [utils.IMAGE_REGIONS[key][0] for key in b19.index if key in utils.IMAGE_REGIONS]
b19 = b19.sort_index()


fig = plt.figure(figsize=(12, 8))

cmap = plt.get_cmap("tab20").resampled(len(b19.columns)) 
disease_colors = {disease: cmap(i) for i, disease in enumerate(utils.causes.values())}

cumval = 0
for i, col in enumerate(utils.causes.values()):
    color = disease_colors.get(col)  
    plt.bar(b19.index, b19[col], bottom=cumval, color=color, label=col)
    cumval = cumval + b19[col]


plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x*1e-3:.0f}k'))
utils.StylizePlot(
    # facecolor="whitesmoke",
    ylabel="Total deaths", 
    title=f"Total attributable deaths in 2019", 
    legend=True, 
    legend_kwargs={"frameon": False},
    xticks_kwargs={"rotation": 90}, 
    despine={"top": True, "right": True}
    )

# plt.savefig(os.path.dirname(wdir) +"/figures/Paper1/Burkart_IMAGEattributableDeaths.png", dpi=300, bbox_inches='tight')

In [ ]:
wdir = "X:/user/liprandicn/mt-comparison/burkart2022/output/ReplicationBurkart"
filename = "mortality_ReplicationBurkart_SSP2_ERA5_1990-2019_*"
region_type="IMAGE"
region = "World"
t_type = "all"
cause = "All causes"
age_group = "All ages"
variable = "mortality"

bur_mean, bur_p025, bur_p975 = utils.LoadMortalityDraws(wdir, filename, region_type, region, t_type, cause, age_group, variable)
print(f"Mean: 1990: {bur_mean.sel(year=1990).values}, 2000: {bur_mean.sel(year=2000).values}, 2010: {bur_mean.sel(year=2010).values}, 2019: {bur_mean.sel(year=2019).values}")
print(f"P2.5: 1990: {bur_p025.sel(year=1990).values}, 2000: {bur_p025.sel(year=2000).values}, 2010: {bur_p025.sel(year=2010).values}, 2019: {bur_p025.sel(year=2019).values}")
print(f"P97.5: 1990: {bur_p975.sel(year=1990).values}, 2000: {bur_p975.sel(year=2000).values}, 2010: {bur_p975.sel(year=2010).values}, 2019: {bur_p975.sel(year=2019).values}")


colors = ["#8C9A9E", "#6F1A07", "#5B9279", "#022B3A", "#E3B448"]
fig, ax = plt.subplots(figsize=(6,5))

plt.plot(bur_mean.year, bur_mean, label="Burkart et al., 2021", linewidth=2, c=colors[0])
plt.fill_between(bur_mean.year, bur_p025, bur_p975, color=colors[0], alpha=0.3)


plt.title(f"{region} {variable} estimations | {t_type.capitalize()} | {age_group.capitalize()} age group")
plt.legend(frameon=False)
plt.ylabel("Excess mortality (thousand people)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

#### Regional values for comparison with paper

In [21]:
# Plot table with Attributable deaths of original 8 countries to compare with paper table
countries = ["BRA", "CHL", "CHN", "COL", "GTM", "MEX", "NZL", "ZAF", "USA"]

wdir = "X:/user/liprandicn/mt-comparison/burkart2022/output/ReplicationBurkart"
filename = "mortality_ReplicationBurkart_SSP2_ERA5_1990-2019_*"
region_type="ISO3"
cause = "All causes"
age_group = "All ages"
variable = "mortality"

mortality_country = defaultdict(lambda: defaultdict(dict))

for region in countries:
    for t_type in ["all", "heat", "cold"]:
        mortality_country[region][t_type] = {}
        mortality_country[region][t_type]["mean"], mortality_country[region][t_type]["p025"],  mortality_country[region][t_type]["p975"] = utils.LoadMortalityDraws(wdir, filename, region_type, region, t_type, cause, age_group, variable)

In [ ]:
region = "BRA"
mortality_country[region]["heat"]["mean"].sel(year=[1990,2019]).values

In [ ]:
bur_mean = xr.open_dataset(wdir+"/mortality_ReplicationBurkart_SSP2_ERA5_1990-2019_1.nc")
bur_mean.sel(t_type="all", region="CHN", region_type="ISO3", age_group="All ages", cause="All causes", var_mor="mean", year=1990).mortality.values.round(0)